# VUR 3-shot Prompting -- GPT-4o and GPT-5

Prompt format mirrors the UTD `generate_prompt()` (single user message, inline examples,
"Example N - Radiology report: ... Correct answer:..." format, generic JSON system prompt).

Examples: one random No-reflux, one random low-grade, one random high-grade report, drawn ONLY
from the train split, redrawn until at least one example has Left reflux and at least one has
Right reflux. The same 3 examples are used for every prompt call.

**API key:** put it in `api_keys.py` in this same folder (see cell 1). Never commit that file.

In [ ]:
import os, json, random, time
import pandas as pd
from openai import OpenAI

from api_keys import OPENAI_API_KEY   # api_keys.py in this folder: OPENAI_API_KEY = "sk-..."

client = OpenAI(api_key=OPENAI_API_KEY)

## Config

In [ ]:
TRAIN_SPLIT_PATH = 'vur_bert_rerun/split_train_v2_split_before_downsample.csv'
EVAL_SPLIT_PATH = 'vur_bert_rerun/split_test_v2_split_before_downsample.csv'

MODELS_TO_RUN = ['gpt-4o', 'gpt-5']
SEED = 42

# Mirrors the UTD system message ("...The result should be stored as key 'answer'.")
SYSTEM_PROMPT = (
    "You are a helpful assistant designed to output JSON. The result should be "
    "stored as keys 'left_grade' and 'right_grade'."
)

CANDIDATE_LABELS = "0.0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0"

LOW_GRADES = {1.0, 1.5, 2.0}          # grades I, I/II, II
HIGH_GRADES = {3.5, 4.0, 4.5, 5.0}    # grades III/IV through V (2/3 and 3 excluded as intermediate)

In [ ]:
# Sanity check -- confirms the split files from the retrain notebook actually exist
# before doing anything else.
from pathlib import Path
for p in [TRAIN_SPLIT_PATH, EVAL_SPLIT_PATH]:
    assert Path(p).exists(), f'{p} not found -- run retrain_vur_bert.ipynb first, or fix the path.'
print('Train and eval split files found.')

## Select fixed 3-shot examples from the train split

In [ ]:
def pick_fixed_examples(train_df, seed=SEED, max_tries=10000):
    """One random No-reflux, one random low-grade, one random high-grade example from train.
    The low/high pair is redrawn until at least one example has Left reflux and at least
    one has Right reflux. If a previous run's examples were saved, checks that they match."""
    rng = random.Random(seed)
    df = train_df.copy()
    df['Left'] = df['Left'].astype(float)
    df['Right'] = df['Right'].astype(float)
    df['max_grade'] = df[['Left', 'Right']].max(axis=1)

    no_reflux = df[df['max_grade'] == 0.0]
    low = df[df['max_grade'].isin(LOW_GRADES)]
    high = df[df['max_grade'].isin(HIGH_GRADES)]
    assert len(no_reflux) and len(low) and len(high), 'An example bin is empty -- check grade bins.'

    nr_ex = no_reflux.loc[rng.choice(list(no_reflux.index))]
    for _ in range(max_tries):
        lo = low.loc[rng.choice(list(low.index))]
        hi = high.loc[rng.choice(list(high.index))]
        has_left = lo['Left'] > 0 or hi['Left'] > 0
        has_right = lo['Right'] > 0 or hi['Right'] > 0
        if has_left and has_right:
            break
    else:
        raise RuntimeError('No low/high pair covers both sides -- check grade bins.')

    examples = [nr_ex, lo, hi]
    example_texts = [str(ex['text']) for ex in examples]
    example_keys = [f"left_grade: {ex['Left']}, right_grade: {ex['Right']}" for ex in examples]

    for ex, key in zip(examples, example_keys):
        print(f"Accession {ex['Accession Number']}: {key}")

    record = [{'accession': str(ex['Accession Number']), 'left': float(ex['Left']),
               'right': float(ex['Right'])} for ex in examples]
    if os.path.exists('vur_3shot_examples.json'):
        saved = json.load(open('vur_3shot_examples.json'))
        assert saved == record, 'Examples differ from the earlier run -- do not resume with these.'
        print('Examples match the earlier run.')
    else:
        with open('vur_3shot_examples.json', 'w') as f:
            json.dump(record, f, indent=2)
    return example_texts, example_keys


train_df = pd.read_csv(TRAIN_SPLIT_PATH)
eval_df = pd.read_csv(EVAL_SPLIT_PATH)
example_texts, example_keys = pick_fixed_examples(train_df)

## Prompt construction and API calls

In [ ]:
def generate_prompt(examples, example_keys, report):
    """Mirrors the UTD generate_prompt() line for line; only the task wording differs."""
    prompt = ("Please read a VCUG report, and decide the grade of vesicoureteral reflux "
              "for the left and right sides. ")
    prompt += (f"You should answer by choosing one of the following labels for each side: "
               f"{CANDIDATE_LABELS} (0.0 means no reflux; 2.5 means grade 2-3, for example). "
               "I'll provide you with three examples first.")
    prompt += f"\n\nExample 1 - Radiology report: \n{examples[0]}\n Correct answer:{example_keys[0]}"
    prompt += f"\n\nExample 2 - Radiology report: \n{examples[1]}\n Correct answer:{example_keys[1]}"
    prompt += f"\n\nExample 3 - Radiology report: \n{examples[2]}\n Correct answer:{example_keys[2]}\n\n"
    prompt += f"Now read the following radiology report:\n\n{report}\n\n"
    prompt += "Now answer: what is the grade of vesicoureteral reflux on the left and right sides?"
    return prompt


def run_inference(client, model, prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                response_format={'type': 'json_object'},
                messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                          {'role': 'user', 'content': prompt}],
            )
            raw = response.choices[0].message.content
            parsed = json.loads(raw)
            return parsed.get('left_grade'), parsed.get('right_grade'), raw
        except Exception as e:
            wait = 2 ** attempt
            print(f'Error ({e}); retry {attempt + 1}/{max_retries} in {wait}s')
            time.sleep(wait)
    return None, None, ''


# Preview one full prompt before spending API calls on 1,040 of them
print(generate_prompt(example_texts, example_keys, eval_df.loc[0, 'text']))

## Run both models on the eval split

In [ ]:
def progress_path(model):
    return f"vur_3shot_{model.replace('-', '_')}_progress.csv"

def load_done(model):
    """Rows already answered successfully. Failed calls (blank grades) are retried."""
    path = progress_path(model)
    if not os.path.exists(path):
        return pd.DataFrame()
    tag = model.replace('-', '_')
    prog = pd.read_csv(path).dropna(subset=[f'{tag}_Left', f'{tag}_Right'])
    return prog.drop_duplicates('Accession Number', keep='last')

assert eval_df['Accession Number'].is_unique, 'Accession numbers must be unique to resume safely'

predictions = {}
for model in MODELS_TO_RUN:
    tag = model.replace('-', '_')
    col_l, col_r, col_raw = f'{tag}_Left', f'{tag}_Right', f'{tag}_raw'
    path = progress_path(model)
    done = load_done(model)
    done_ids = set(done['Accession Number']) if len(done) else set()
    todo = eval_df[~eval_df['Accession Number'].isin(done_ids)]
    print(f'{model}: {len(done_ids)} already done, {len(todo)} to go')

    for n, (_, row) in enumerate(todo.iterrows(), 1):
        prompt = generate_prompt(example_texts, example_keys, row['text'])
        l, r, raw = run_inference(client, model, prompt)
        pd.DataFrame([{'Accession Number': row['Accession Number'], col_l: l, col_r: r, col_raw: raw}]) \
            .to_csv(path, mode='a', header=not os.path.exists(path), index=False)   # saved immediately
        if n % 25 == 0:
            print(f'  {model}: {len(done_ids) + n}/{len(eval_df)}')

    final = load_done(model)[['Accession Number', col_l, col_r, col_raw]]
    eval_df = eval_df.drop(columns=[c for c in (col_l, col_r, col_raw) if c in eval_df.columns]) \
                     .merge(final, on='Accession Number', how='left')
    predictions[model] = (col_l, col_r)

eval_df.to_csv('vur_3shot_predictions.csv', index=False)
print('Saved vur_3shot_predictions.csv')

## Metrics: Wilson per-label, macro/weighted + patient-level bootstrap, ordinal (QWK/MAE)

Self-contained -- no external imports beyond sklearn/numpy/pandas, so this section runs the same way in either notebook.

In [ ]:
import math
import random
import numpy as np
from sklearn.metrics import cohen_kappa_score

SEVERITIES = ['0.0', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0', '4.5', '5.0']
SEVERITY_RANK = {v: i for i, v in enumerate([float(s) for s in SEVERITIES])}
VMETRICS = ['precision', 'recall', 'f1', 'accuracy']


def wilson_ci(x, n, z=1.96):
    """95% Wilson score interval for a binomial proportion x/n."""
    if n <= 0:
        return (0.0, 0.0)
    phat = x / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (z * math.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def per_class_metrics(tp, fp, fn, tn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    return precision, recall, f1, accuracy

In [ ]:
def build_label_confusion(df, lcol, rcol, true_lcol='Left', true_rcol='Right'):
    """20 one-vs-rest confusion matrices (10 severities x 2 sides) from patient rows."""
    d = {}
    for sev in SEVERITIES:
        sev_f = float(sev)
        for side, true_col, pred_col in [('Left', true_lcol, lcol), ('Right', true_rcol, rcol)]:
            true_vals = df[true_col].astype(float).values
            pred_vals = df[pred_col].astype(float).values
            tp = fp = fn = tn = 0
            for t, p in zip(true_vals, pred_vals):
                if t == sev_f and p == sev_f:
                    tp += 1
                elif t != sev_f and p == sev_f:
                    fp += 1
                elif t == sev_f and p != sev_f:
                    fn += 1
                else:
                    tn += 1
            d[f'{sev}_{side}'] = dict(tp=tp, fp=fp, fn=fn, tn=tn)
    return d


def per_label_all_metrics(d_raw):
    out = {}
    for label, counts in d_raw.items():
        tp, fp, fn, tn = counts['tp'], counts['fp'], counts['fn'], counts['tn']
        p, r, f1, a = per_class_metrics(tp, fp, fn, tn)
        out[label] = dict(precision=(p, tp, tp + fp), recall=(r, tp, tp + fn),
                           f1=(f1, 2 * tp, 2 * tp + fp + fn), accuracy=(a, tp + tn, tp + tn + fp + fn))
    return out


def vcug_aggregate_across_labels(d_raw, labels):
    per_class = {}
    for label in labels:
        c = d_raw[label]
        tp, fp, fn, tn = c['tp'], c['fp'], c['fn'], c['tn']
        p, r, f1, a = per_class_metrics(tp, fp, fn, tn)
        per_class[label] = dict(precision=p, recall=r, f1=f1, accuracy=a, support=tp + fn)
    macro, weighted = {}, {}
    total = sum(per_class[l]['support'] for l in labels)
    for key in ['precision', 'recall', 'accuracy']:
        macro[key] = sum(per_class[l][key] for l in labels) / len(labels)
        weighted[key] = (sum(per_class[l][key] * per_class[l]['support'] for l in labels) / total) if total > 0 else 0
    macro['f1'] = (2 * macro['precision'] * macro['recall']) / (macro['precision'] + macro['recall']) if (macro['precision'] + macro['recall']) > 0 else 0
    weighted['f1'] = (2 * weighted['precision'] * weighted['recall']) / (weighted['precision'] + weighted['recall']) if (weighted['precision'] + weighted['recall']) > 0 else 0
    return macro, weighted


def patient_bootstrap_ci(df, lcol, rcol, true_lcol='Left', true_rcol='Right', n_boot=2000, seed=0):
    rng = random.Random(seed)
    n = len(df)
    boot = {'macro': {m: [] for m in VMETRICS}, 'weighted': {m: [] for m in VMETRICS}}
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]
        sample = df.iloc[idx].reset_index(drop=True)
        d_boot = build_label_confusion(sample, lcol, rcol, true_lcol, true_rcol)
        macro, weighted = vcug_aggregate_across_labels(d_boot, list(d_boot.keys()))
        for m in VMETRICS:
            boot['macro'][m].append(macro[m])
            boot['weighted'][m].append(weighted[m])
    def pct(vals):
        vals = sorted(vals)
        lo = vals[int(0.025 * len(vals))]
        hi = vals[min(len(vals) - 1, int(0.975 * len(vals)))]
        return lo, hi
    return {conv: {m: pct(boot[conv][m]) for m in VMETRICS} for conv in ['macro', 'weighted']}


def ordinal_metrics(true_vals, pred_vals):
    """Quadratic-weighted kappa + mean absolute error in grade units."""
    true_ranks = [SEVERITY_RANK[float(v)] for v in true_vals]
    pred_ranks = [SEVERITY_RANK[float(v)] for v in pred_vals]
    qwk = cohen_kappa_score(true_ranks, pred_ranks, weights='quadratic')
    mae = np.mean(np.abs(np.array(true_vals, dtype=float) - np.array(pred_vals, dtype=float)))
    return qwk, mae


def print_full_metrics(df, lcol, rcol, model_name, true_lcol='Left', true_rcol='Right', n_boot=2000):
    """Runs the complete metric suite and prints a manuscript-style summary."""
    print(f'{"="*70}\n{model_name}\n{"="*70}')

    d_raw = build_label_confusion(df, lcol, rcol, true_lcol, true_rcol)
    per_label = per_label_all_metrics(d_raw)

    print('\n-- Wilson per-label (precision) --')
    for label, m in per_label.items():
        val, x, n = m['precision']
        lo, hi = wilson_ci(x, n)
        print(f'  {label:12s} precision = {val:.4f} [{lo:.4f}, {hi:.4f}]  (n={n})')

    macro, weighted = vcug_aggregate_across_labels(d_raw, list(d_raw.keys()))
    boot_ci = patient_bootstrap_ci(df, lcol, rcol, true_lcol, true_rcol, n_boot=n_boot)

    print('\n-- Macro / weighted aggregate (patient-level bootstrap) --')
    for conv, point in [('macro', macro), ('weighted', weighted)]:
        for metric in VMETRICS:
            lo, hi = boot_ci[conv][metric]
            print(f'  {conv:8s} {metric:10s} = {point[metric]:.4f} [{lo:.4f}, {hi:.4f}]')

    print('\n-- Ordinal metrics (QWK, MAE in grade units) --')
    for side, true_col, pred_col in [('Left', true_lcol, lcol), ('Right', true_rcol, rcol)]:
        qwk, mae = ordinal_metrics(df[true_col].values, df[pred_col].values)
        print(f'  {side:6s} QWK = {qwk:.4f}   MAE = {mae:.4f}')
    true_both = list(df[true_lcol].values) + list(df[true_rcol].values)
    pred_both = list(df[lcol].values) + list(df[rcol].values)
    qwk_c, mae_c = ordinal_metrics(true_both, pred_both)
    print(f'  {"Both":6s} QWK = {qwk_c:.4f}   MAE = {mae_c:.4f}')

    return {'per_label': per_label, 'macro': macro, 'weighted': weighted,
            'bootstrap_ci': boot_ci, 'd_raw': d_raw}

## Run the full metric suite for each model

In [ ]:
VALID_GRADES = {0.0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0}

def to_valid_grade(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if v in VALID_GRADES else None

all_results = {}
for model, (col_l, col_r) in predictions.items():
    df_m = eval_df.copy()
    df_m[col_l] = df_m[col_l].map(to_valid_grade)
    df_m[col_r] = df_m[col_r].map(to_valid_grade)
    bad = df_m[[col_l, col_r]].isnull().any(axis=1)
    print(f'{model}: {bad.sum()} / {len(df_m)} rows missing or off-scale (excluded from metrics)')
    if bad.any():
        print(eval_df.loc[bad, [col_l, col_r]].head(10))
    all_results[model] = print_full_metrics(df_m[~bad].reset_index(drop=True),
                                            col_l, col_r, f'{model} (3-shot)',
                                            true_lcol='Left', true_rcol='Right', n_boot=2000)